## 1. Imports

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer

## 2. Chargement des données

In [16]:
df = pd.read_csv(r"../02-preprocessing/movies_preprocessed_clean.csv", index_col=0)
cast = pd.read_csv(r'../02-preprocessing/cast_group_clean.csv', index_col=0)

Je vais merge les 2 DF voir si beaucoup de NA etc

In [17]:
df_cast = pd.merge(df, cast, how='inner', on='tconst')

In [18]:
print(f'df.shape {df.shape}')
print(f'df.cast.shape {cast.shape}')
print(f'df_cast.shape {df_cast.shape}')

df.shape (330702, 33)
df.cast.shape (682132, 2)
df_cast.shape (328561, 35)


On ne perd que 2000 films par rapport au df standard, négligeable comparé à l'importance de la feature,
on va donc essayer de produire une matrice pour entrainer un modele de CountVectorizer

Je vais crée une liste de tout les genre avant tout 

In [20]:
genre_cols = df_cast.loc[:, 'Action':'Western'].columns.tolist()

Et j'instancie le CV

In [27]:
CV = CountVectorizer()

cast_sparse = CV.fit_transform(df_cast['clean_name'])
genre_sparse = csr_matrix(df_cast[genre_cols].values)

J'ai vu que hstack permettait de coller 2 matrices, et tocsr le convertit dans un format qui permet de découper une seule ligne

In [28]:
matrice = hstack([cast_sparse, genre_sparse]).tocsr()

In [46]:
position = 0
scores = cosine_similarity(matrice[position], matrice)

scores = scores.flatten()
ordre = scores.argsort()[::-1]
top = ordre[1:11]

J'englobe tout ca dans une fonction

In [58]:
def recherche_par_titre(titre, n=10):
    matches = df_cast[df_cast['primaryTitle'].str.contains(titre, case=False, na=False)]
    if matches.empty:
        return f"Aucun film trouvé pour '{titre}'"
    position = matches.sort_values('weight_rating', ascending=False).index[0]
    
    sims = cosine_similarity(matrice[position], matrice).flatten()
    ordre = sims.argsort()[::-1]
    top = ordre[1:n+1]
    return df_cast.iloc[top][['primaryTitle', 'startYear', 'clean_name', 'weight_rating']]


In [62]:
recherche_par_titre('ring')

,primaryTitle,startYear,clean_name,weight_rating
77489,The Lord of the Rings: The Two Towers,2002.0,elijahwood ianmckellen viggomortensen orlandob...,8.791639
65402,The Lord of the Rings: The Fellowship of the Ring,2001.0,elijahwood ianmckellen orlandobloom seanbean a...,8.892303
192279,Montagna con Forza,2002.0,flintjuventinobeppe,6.162411
233410,The Once,2023.0,sockswhitmore scotthillman,6.143652
147744,Third World,2022.0,sabanuraksoy cangüvenç metinkuru,6.157328
292155,El cruce de la pampa,2016.0,rolyserrano gonzalourtizberéa davidbisbano,6.157242
85244,The King's Musketeers,1957.0,josephlerner,6.155575
7342,Chandu on the Magic Island,1935.0,ironeyescody,6.096523
319064,Barzaj,2018.0,alejandrogonzálezsalgado,6.161134
268967,Mis Quejas hacia Dios,2011.0,jesúsdelogroño,6.151916


On a une part de pertinence, mais une fois cette pertinence épuisée le résultat s'effondre : on part dans des films complètement inconnus qui ne partagent qu'un genre. 